# 09 · Auswertung zu vorhandenen Messungen ergänzen

**Run All ist eine lokale Vorschau.** Dieses Notebook benötigt Notebook 08 nicht im selben Kernel.
Trage in `config/gasi_analysis.json` die tatsächlichen Eingabedataset-IDs, relativen Dataset-Dateipfade,
Original-Hashes und ausgewählten Record-/Stufenbereiche ein. Das serverseitige `source_receipt.json` der Messung
und die lokale `verification.json` aus Notebook 08 liefern diese Angaben.

Jede eigenständige Auswertung erhält eine eigene `analysis_id`. Sie kann mehrere Messungen verwenden.
Nur `gasi_workflow.targets.analysis` wird benötigt. Ergebnisse und das vollständige Auswertemanifest werden
als ATTACHMENT abgelegt. Native Dataset-Parents sind standardmäßig deaktiviert; sie benötigen einen
bestätigten Instanznachweis im zentralen Zielabschnitt.

Die letzte Zelle startet das Schreiben erst nach bewusster Aktivierung. Eingabedaten werden ausschließlich
gelesen, heruntergeladen und gegen Quellmanifest, Auswahl und SHA-256 geprüft.
Sie werden weder neu hochgeladen noch überschrieben.


In [ ]:
from pathlib import Path
import importlib
import gasi_analysis
from IPython.display import FileLink, display

gasi_analysis = importlib.reload(gasi_analysis)
path = Path("config/gasi_analysis.json")
if not path.exists():
    with path.open("xb") as target:
        target.write(Path("config/gasi_analysis.example.json").read_bytes())
config = gasi_analysis.runs.load_run(path)
issues = gasi_analysis.validate_analysis(config)
targets, message = gasi_analysis.ob.load_target_config()
print(message)
try:
    target = gasi_analysis.runs._targets(targets, ("analysis",))["analysis"]
    print("Auswertungsziel:", gasi_analysis.ob.target_path(target))
except gasi_analysis.ob.CheckFailed as exc:
    issues.append(str(exc))
preview_sha256 = None
if issues:
    for issue in issues:
        print("-", issue)
else:
    bundle = gasi_analysis.prepare_analysis(config)
    preview_sha256 = bundle["sha256"]
    print("Lokale Vorschau geprüft; Eingaben und Rechte werden vor dem Schreiben live erneut geprüft.")
    for item in config["inputs"]:
        print("Eingabe:", item["measurement_id"], item["dataset_id"], item["file_path"])
    print("Ergebnisdateien:", len(bundle["entries"]), "; Schreibzugriff deaktiviert.")


In [ ]:
UPLOAD_ENABLED = False
CONFIRMATION = ""  # UPLOAD GASI ANALYSIS TO PLAYGROUND
result, message = gasi_analysis.upload_analysis(
    enabled=UPLOAD_ENABLED, confirmation=CONFIRMATION,
    expected_manifest_sha=globals().get("preview_sha256"))
print(message)
if result:
    print(result)
